In [ ]:
# ============================================================
# 01_fetch_historical.py
# Fetch complete historical data by rolling through start/end Unix timestamps
# ============================================================

import requests, json, time
import pandas as pd
from datetime import datetime

BASE = "https://prices.curve.fi"

def fetch_snapshots_window(address: str, start_ts: int, end_ts: int) -> list:
    r = requests.get(
        f"{BASE}/v1/crvusd/markets/ethereum/{address}/snapshots",
        params={"start": start_ts, "end": end_ts, "per_page": 200},
        timeout=20
    )
    return r.json().get('data', [])

def fetch_full_history(address: str, symbol: str) -> pd.DataFrame:
    """
    Fetch data from 2023-06-01 (crvUSD launch date) up to now
    Using 100-day windows with 100-day steps
    """
    all_records = []
    seen_dates = set()
   
    # crvUSD launched in June 2023
    start = int(datetime(2023, 6, 1).timestamp())
    end = int(datetime(2026, 5, 10).timestamp())
   
    window = 100 * 86400  # 100 days
    cur = start
   
    while cur < end:
        win_end = min(cur + window, end)
        batch = fetch_snapshots_window(address, cur, win_end)
       
        new = 0
        for s in batch:
            dt = s.get('dt', '')
            if dt and dt not in seen_dates:
                seen_dates.add(dt)
                all_records.append(s)
                new += 1
       
        print(f" {datetime.fromtimestamp(cur).date()} → "
              f"{datetime.fromtimestamp(win_end).date()}: "
              f"{len(batch)} raw records / {new} new records (total: {len(all_records)})")
       
        cur += window
        time.sleep(0.3)
   
    if not all_records:
        return pd.DataFrame()
   
    df = pd.DataFrame(all_records)
    df['dt'] = pd.to_datetime(df['dt'])
    df = df.sort_values('dt').drop_duplicates('dt').set_index('dt')
    return df


# ── Fetch All Markets ──────────────────────────────────────────
r = requests.get(f"{BASE}/v1/crvusd/markets?chain=ethereum")
markets = r.json()['chains']['ethereum']['data']

all_dfs = {}
for m in markets:
    sym = m.get('collateral_token', {}).get('symbol', 'unknown')
    addr = m['address']
    print(f"\n{'='*50}")
    print(f"{sym} ({addr[:10]}...)")
   
    df = fetch_full_history(addr, sym)
    all_dfs[sym] = df
   
    if not df.empty:
        print(f" ✓ Final: {len(df)} records [{df.index.min().date()} → {df.index.max().date()}]")


# ── Save Files ──────────────────────────────────────────────────
for sym, df in all_dfs.items():
    if not df.empty:
        # Add suffix to distinguish markets with the same name
        fname = f"hist_{sym}.csv"
        if fname in [f"hist_{s}.csv" for s in list(all_dfs.keys())[:list(all_dfs.keys()).index(sym)]]:
            fname = f"hist_{sym}_2.csv"
        
        df.to_csv(fname)
        print(f"Saved: {fname} ({len(df)} rows)")

print("\nCompleted")

In [ ]:
# ============================================================
# 02_rmt_analysis.py
# Complete RMT Correction + Flash Crash Event Validation + Plotting
# Ready to run — no parameters need to be changed
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import FancyArrowPatch
import warnings

warnings.filterwarnings('ignore')


# ══════════════════════════════════════════════════════════════
# PART 1: Load Data & Build 8-Dimensional Scoring System
# ══════════════════════════════════════════════════════════════

def load_and_score(symbols: list) -> pd.DataFrame:
    """
    Load CSV files, calculate dimension scores for each market,
    and align them to the same timeline.
    
    Returns: MultiIndex DataFrame with columns = (symbol, dimension)
    """
    all_scores = {}
    for sym in symbols:
        try:
            df = pd.read_csv(f'hist_{sym}.csv', index_col='dt', parse_dates=True)
        except FileNotFoundError:
            print(f"  Skipping {sym}: File not found")
            continue
        
        s = pd.DataFrame(index=df.index)

        # ── Dimension 1: Debt Ceiling Utilization (Weight 30%) ──
        # utilization = debt / (debt + borrowable)
        borrowable = df['borrowable'].clip(lower=0)
        total_debt = df['total_debt'].clip(lower=0)
        util = total_debt / (total_debt + borrowable + 1e-10)
        s['s_debt_ceiling'] = score_decreasing(util, good=0.50, bad=0.90)

        # ── Dimension 2: Collateral Ratio (Weight 5%) ──
        col_usd = df['total_collateral_usd'].clip(lower=0)
        debt_usd = df['total_debt_usd'].clip(lower=1)
        col_ratio = col_usd / debt_usd
        s['s_collateral_ratio'] = score_increasing(col_ratio, good=2.0, bad=1.15)

        # ── Dimension 3: Soft Liquidation Exposure (Weight 10%) ──
        stbl_usd = df['total_stablecoin_usd'].clip(lower=0)
        soft_ratio = stbl_usd / (col_usd + 1e-10)
        s['s_soft_liq'] = score_decreasing(soft_ratio, good=0.02, bad=0.25)

        # ── Dimension 4: Bad Debt Proxy (Weight 10%) ──
        bad_debt_proxy = (debt_usd - col_usd).clip(lower=0) / (debt_usd + 1e-10)
        s['s_bad_debt'] = score_decreasing(bad_debt_proxy, good=0.0, bad=0.05)

        # ── Dimension 5: Price Momentum (Weight 15%) ──
        price = df['amm_price'].clip(lower=1e-10)
        ret7 = price.pct_change(7)
        ret30 = price.pct_change(30)
        momentum = ret7 - ret30  # Positive = short-term stronger = healthier
        s['s_momentum'] = score_increasing(momentum, good=0.02, bad=-0.20)

        # ── Dimension 6: Price Volatility (Weight 15%) ──
        vol = price.pct_change().rolling(30).std()
        s['s_volatility'] = score_decreasing(vol, good=0.015, bad=0.055)

        # ── Dimension 7: Borrower Concentration (Weight 5%) ──
        sd2 = df['sum_debt_squared'].clip(lower=0)
        hhi = sd2 / (total_debt**2 + 1e-10)
        hhi_norm = hhi.clip(0, 1)
        s['s_concentration'] = score_decreasing(hhi_norm, good=0.01, bad=0.15)

        # ── Dimension 8: Soft Liquidation Efficiency (Weight 10%) ──
        borrow_apy = df['borrow_apy'].clip(lower=0) / 100.0
        s['s_soft_efficiency'] = score_decreasing(borrow_apy, good=0.05, bad=0.40)

        all_scores[sym] = s

    # Align all markets to common time index (intersection)
    common_idx = all_scores[list(all_scores.keys())[0]].index
    for sym in all_scores:
        common_idx = common_idx.intersection(all_scores[sym].index)

    aligned = pd.concat(
        {sym: df.reindex(common_idx) for sym, df in all_scores.items()},
        axis=1
    )
    return aligned


def score_decreasing(series, good, bad):
    """Lower value = healthier: 1 below 'good', 0 above 'bad'"""
    return ((bad - series) / (bad - good + 1e-12)).clip(0, 1)


def score_increasing(series, good, bad):
    """Higher value = healthier: 1 above 'good', 0 below 'bad'"""
    return ((series - bad) / (good - bad + 1e-12)).clip(0, 1)


# Weights for the 8 dimensions (in order)
WEIGHTS = np.array([0.30, 0.05, 0.10, 0.10, 0.15, 0.15, 0.05, 0.10])

DIM_NAMES = ['debt_ceiling', 'collateral_ratio', 'soft_liq', 'bad_debt',
             'momentum', 'volatility', 'concentration', 'soft_efficiency']

# Markets with complete history
SYMBOLS = ['WETH', 'WBTC', 'wstETH', 'tBTC', 'sfrxETH']

print("Loading data and calculating dimension scores...")
scores_multi = load_and_score(SYMBOLS)
print(f"Time range: {scores_multi.index.min().date()} → {scores_multi.index.max().date()}")
print(f"Total {len(scores_multi)} trading days, {len(SYMBOLS)} markets")

In [ ]:
# ============================================================
# Complete Combined Script - Ready to Run in One Cell
# ============================================================



import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings

warnings.filterwarnings('ignore')


# ── Scoring Functions ───────────────────────────────────────
def score_decreasing(series, good, bad):
    """Lower value = healthier"""
    return ((bad - series) / (bad - good + 1e-12)).clip(0, 1)

def score_increasing(series, good, bad):
    """Higher value = healthier"""
    return ((series - bad) / (good - bad + 1e-12)).clip(0, 1)


WEIGHTS = np.array([0.30, 0.05, 0.10, 0.10, 0.15, 0.15, 0.05, 0.10])

DIM_NAMES = ['debt_ceiling', 'collateral_ratio', 'soft_liq', 'bad_debt',
             'momentum', 'volatility', 'concentration', 'soft_efficiency']

SYMBOLS = ['WETH', 'WBTC', 'wstETH', 'tBTC', 'sfrxETH']


# ── PART 1: Load Data + Scoring ─────────────────────────────
def load_and_score(symbols):
    all_scores = {}
    for sym in symbols:
        try:
            df = pd.read_csv(f'hist_{sym}.csv', index_col='dt', parse_dates=True)
        except FileNotFoundError:
            print(f"  Skipping {sym}")
            continue
        
        s = pd.DataFrame(index=df.index)

        # Dimension 1: Debt Ceiling Utilization
        borrowable = df['borrowable'].clip(lower=0)
        total_debt = df['total_debt'].clip(lower=0)
        util = total_debt / (total_debt + borrowable + 1e-10)
        s['s_debt_ceiling'] = score_decreasing(util, good=0.50, bad=0.90)

        # Dimension 2: Collateral Ratio
        col_usd = df['total_collateral_usd'].clip(lower=0)
        debt_usd = df['total_debt_usd'].clip(lower=1)
        s['s_collateral_ratio'] = score_increasing(col_usd / debt_usd, good=2.0, bad=1.15)

        # Dimension 3: Soft Liquidation Exposure
        stbl_usd = df['total_stablecoin_usd'].clip(lower=0)
        s['s_soft_liq'] = score_decreasing(stbl_usd / (col_usd + 1e-10), good=0.02, bad=0.25)

        # Dimension 4: Bad Debt Proxy
        bad_proxy = (debt_usd - col_usd).clip(lower=0) / (debt_usd + 1e-10)
        s['s_bad_debt'] = score_decreasing(bad_proxy, good=0.0, bad=0.05)

        # Dimension 5: Price Momentum
        price = df['amm_price'].clip(lower=1e-10)
        momentum = price.pct_change(7) - price.pct_change(30)
        s['s_momentum'] = score_increasing(momentum, good=0.02, bad=-0.20)

        # Dimension 6: Volatility
        vol = price.pct_change().rolling(30).std()
        s['s_volatility'] = score_decreasing(vol, good=0.015, bad=0.055)

        # Dimension 7: Borrower Concentration
        sd2 = df['sum_debt_squared'].clip(lower=0)
        hhi = (sd2 / (total_debt**2 + 1e-10)).clip(0, 1)
        s['s_concentration'] = score_decreasing(hhi, good=0.01, bad=0.15)

        # Dimension 8: Soft Liquidation Efficiency
        borrow_apy = df['borrow_apy'].clip(lower=0) / 100.0
        s['s_soft_efficiency'] = score_decreasing(borrow_apy, good=0.05, bad=0.40)

        all_scores[sym] = s

    # Align all markets to common dates
    common_idx = list(all_scores.values())[0].index
    for s in all_scores.values():
        common_idx = common_idx.intersection(s.index)

    return pd.concat(
        {sym: df.reindex(common_idx) for sym, df in all_scores.items()}, axis=1)


print("Loading data...")
scores_multi = load_and_score(SYMBOLS)
print(f"Time range: {scores_multi.index.min().date()} → {scores_multi.index.max().date()}")


# ── PART 2: System Matrix (Cross-Market Average per Dimension) ─────
def build_system_matrix(scores_multi, symbols, dim_names):
    rows = {}
    for dim in dim_names:
        cols = [scores_multi[(sym, f's_{dim}')]
                for sym in symbols
                if (sym, f's_{dim}') in scores_multi.columns]
        if cols:
            rows[dim] = pd.concat(cols, axis=1).mean(axis=1)
    return pd.DataFrame(rows).dropna()


system_scores = build_system_matrix(scores_multi, SYMBOLS, DIM_NAMES)
print(f"System matrix shape: {system_scores.shape}")


# ── PART 3: RMT Analysis ─────────────────────────────────────
def marchenko_pastur_upper(T, N):
    q = T / N
    return (1 + 1 / np.sqrt(q)) ** 2


def compute_effective_rank(eigenvalues):
    eigs = eigenvalues[eigenvalues > 1e-10]
    p = eigs / eigs.sum()
    return np.exp(-np.sum(p * np.log(p + 1e-15)))

In [ ]:
def rmt_clean_and_rank(window_df):
    T, N = window_df.shape
    if T < N + 5:
        return np.nan, 0
   
    # Filter out columns with zero variance (constant columns)
    std = window_df.std()
    valid_cols = std[std > 1e-8].index
    window_df = window_df[valid_cols]
    T, N = window_df.shape
    if N < 2:
        return np.nan, 0
   
    # Standardize the data
    X = (window_df - window_df.mean()) / window_df.std()
   
    # Drop any rows containing NaN
    X = X.dropna()
    T, N = X.shape
    if T < N + 5:
        return np.nan, 0
   
    # Compute correlation matrix, handling numerical issues
    C = X.corr().values
   
    # Check for NaN/Inf values
    if not np.all(np.isfinite(C)):
        C = np.nan_to_num(C, nan=0.0, posinf=1.0, neginf=-1.0)
        np.fill_diagonal(C, 1.0)
   
    # Force symmetry (eliminate floating-point errors)
    C = (C + C.T) / 2
   
    # Use SVD instead of eigh for better numerical stability
    try:
        eigs = np.linalg.eigvalsh(C)
    except np.linalg.LinAlgError:
        try:
            # Fallback: SVD
            _, s, _ = np.linalg.svd(C)
            eigs = s
        except Exception:
            return np.nan, 0
   
    eigs = np.sort(np.abs(eigs))[::-1]  # Take absolute values to avoid negative eigenvalues
   
    lam_plus = marchenko_pastur_upper(T, N)
    n_signal = int(np.sum(eigs > lam_plus))
    eigs_clean = eigs.copy()
    noise_mask = eigs <= lam_plus
    if noise_mask.sum() > 0:
        eigs_clean[noise_mask] = eigs[noise_mask].mean()
   
    return compute_effective_rank(eigs_clean), n_signal


def rolling_rmt(system_scores, window=60):
    results = []
    failed = 0
   
    for i in range(window, len(system_scores)):
        t = system_scores.index[i]
        win = system_scores.iloc[i-window:i]
        cur = system_scores.iloc[i].values
       
        try:
            eff_rank, n_signal = rmt_clean_and_rank(win)
        except Exception as e:
            eff_rank, n_signal = np.nan, 0
            failed += 1
       
        results.append({
            'date': t,
            'health_orig': float(np.nansum(cur * WEIGHTS)),
            'eff_rank': eff_rank,
            'n_signal': n_signal,
            **{f's_{dim}': float(cur[j]) for j, dim in enumerate(DIM_NAMES)}
        })
   
    df = pd.DataFrame(results).set_index('date')
   
    # Interpolate a small number of NaN values
    df['eff_rank'] = df['eff_rank'].interpolate(method='linear').fillna(method='bfill')
   
    if failed > 0:
        print(f"  Warning: {failed} time points failed calculation, filled by interpolation")
   
    # Calculate normal period baseline (2024)
    normal_mask = (df.index >= '2024-01-01') & (df.index <= '2024-12-31')
    eff_rank_normal = df.loc[normal_mask, 'eff_rank'].median()
   
    df['eff_rank_normal'] = eff_rank_normal
    df['correction'] = (df['eff_rank'] / eff_rank_normal).clip(upper=1.0)
    df['health_adj'] = df['health_orig'] * df['correction']
   
    print(f"Baseline Effective Rank (2024 median): {eff_rank_normal:.3f}")
    print(f"Effective Rank Range: {df['eff_rank'].min():.3f} ~ {df['eff_rank'].max():.3f}")
    return df, eff_rank_normal


print("\nRunning Rolling RMT calculation...")
results, eff_rank_normal = rolling_rmt(system_scores, window=60)
print(f"Result rows: {len(results)}")
print(results[['health_orig', 'health_adj', 'eff_rank', 'correction']].tail(3))

In [ ]:
# ── Event Analysis Table + Key Metrics + Plotting (continue from previous code) ──────────────

EVENT_DATE = pd.Timestamp('2025-10-10 21:36:00')

# ── Event Analysis Table ─────────────────────────────────────
print("\n" + "="*75)
print(f"{'Date':12s} {'Original Health':>12s} {'Adjusted Health':>12s} "
      f"{'Eff. Rank':>10s} {'Correction':>9s} {'Signal Dim':>10s}")
print("-"*75)

for d in [-45, -30, -21, -14, -7, -3, -1, 0, 3, 7, 14]:
    target = EVENT_DATE + pd.Timedelta(days=d)
    idx = results.index.get_indexer([target], method='nearest')[0]
    row = results.iloc[idx]
    marker = " ◄ EVENT" if d == 0 else ""
    
    print(f"{str(results.index[idx].date()):12s} "
          f"{row['health_orig']:12.3f} {row['health_adj']:12.3f} "
          f"{row['eff_rank']:10.3f} {row['correction']:9.3f} "
          f"{int(row['n_signal']):10d}{marker}")

# ── Key Metrics ─────────────────────────────────────────────
normal = results.loc['2024-01-01':'2024-12-31']
pre14 = results.loc[EVENT_DATE - pd.Timedelta(days=14):EVENT_DATE - pd.Timedelta(days=1)]
ev = results.iloc[results.index.get_indexer([EVENT_DATE], method='nearest')[0]]

print(f"\nNormal Period 2024: eff_rank={normal['eff_rank'].mean():.3f} "
      f"health_orig={normal['health_orig'].mean():.3f}")
print(f"14 Days Before Event: eff_rank={pre14['eff_rank'].mean():.3f} "
      f"health_orig={pre14['health_orig'].mean():.3f} "
      f"health_adj={pre14['health_adj'].mean():.3f} "
      f"diff={pre14['health_orig'].mean() - pre14['health_adj'].mean():.3f}")
print(f"Event Day: eff_rank={ev['eff_rank']:.3f} "
      f"health_orig={ev['health_orig']:.3f} "
      f"health_adj={ev['health_adj']:.3f} "
      f"penalty={(1 - ev['correction'])*100:.1f}%")

# ── Early Warning Analysis ───────────────────────────────────
thresh = 0.5
window_start = EVENT_DATE - pd.Timedelta(days=60)
window_end = EVENT_DATE + pd.Timedelta(days=30)
window_res = results.loc[window_start:window_end]

orig_breach = window_res.loc[window_res['health_orig'] < thresh].index
adj_breach = window_res.loc[window_res['health_adj'] < thresh].index

if len(orig_breach) and len(adj_breach):
    lead = (orig_breach[0] - adj_breach[0]).days
    print(f"Early Warning: Adjusted score breached 0.5 threshold "
          f"{abs(lead)} days earlier than original score")
elif len(adj_breach) and not len(orig_breach):
    print(f"Early Warning: Original score never breached 0.5, "
          f"but adjusted score did on {adj_breach[0].date()}")
else:
    print("Early Warning: Neither score breached 0.5 threshold in this window "
          "(please check threshold)")
    # Fallback: show lowest points
    orig_min = window_res['health_orig'].min()
    adj_min = window_res['health_adj'].min()
    orig_min_date = window_res['health_orig'].idxmin()
    adj_min_date = window_res['health_adj'].idxmin()
    print(f"  Original score lowest: {orig_min:.3f} @ {orig_min_date.date()}")
    print(f"  Adjusted score lowest: {adj_min:.3f} @ {adj_min_date.date()}")

In [ ]:
# ── Verify Event Details from August 2024 ─────────────────────────────────

aug2024_start = pd.Timestamp('2024-07-01')
aug2024_end = pd.Timestamp('2025-03-01')
aug_window = results.loc[aug2024_start:aug2024_end]

print("Key Metrics for July–September 2024 Window:")
print(f"{'Date':12s} {'Original Health':>12s} {'Adjusted Health':>12s} "
      f"{'Eff. Rank':>10s} {'Correction':>9s}")
print("-"*55)

# Print every 3 days for readability
for date, row in aug_window.iloc[::3].iterrows():
    print(f"{str(date.date()):12s} {row['health_orig']:12.3f} "
          f"{row['health_adj']:12.3f} {row['eff_rank']:10.3f} "
          f"{row['correction']:9.3f}")

# Find the date with the largest gap between original and adjusted health
aug_window['gap'] = aug_window['health_orig'] - aug_window['health_adj']
max_gap_idx = aug_window['gap'].idxmax()
max_gap_row = aug_window.loc[max_gap_idx]

print(f"\nLargest Gap Point: {max_gap_idx.date()}")
print(f"  Original Health: {max_gap_row['health_orig']:.3f}")
print(f"  Adjusted Health: {max_gap_row['health_adj']:.3f}")
print(f"  Gap: {max_gap_row['gap']:.3f}")
print(f"  Effective Rank: {max_gap_row['eff_rank']:.3f}")
print(f"  Penalty: {(1 - max_gap_row['correction'])*100:.1f}%")

In [ ]:
# ── Verify Event Details from February–April 2024 ─────────────────────────────

aug2024_start = pd.Timestamp('2024-02-01')
aug2024_end = pd.Timestamp('2024-04-15')
aug_window = results.loc[aug2024_start:aug2024_end]

print("Key Metrics for February–April 2024 Window:")
print(f"{'Date':12s} {'Original Health':>12s} {'Adjusted Health':>12s} "
      f"{'Eff. Rank':>10s} {'Correction':>9s}")
print("-"*55)

# Print every 3 days for readability
for date, row in aug_window.iloc[::3].iterrows():
    print(f"{str(date.date()):12s} {row['health_orig']:12.3f} "
          f"{row['health_adj']:12.3f} {row['eff_rank']:10.3f} "
          f"{row['correction']:9.3f}")

# Find the date with the largest gap between original and adjusted health
aug_window['gap'] = aug_window['health_orig'] - aug_window['health_adj']
max_gap_idx = aug_window['gap'].idxmax()
max_gap_row = aug_window.loc[max_gap_idx]

print(f"\nLargest Gap Point: {max_gap_idx.date()}")
print(f"  Original Health: {max_gap_row['health_orig']:.3f}")
print(f"  Adjusted Health: {max_gap_row['health_adj']:.3f}")
print(f"  Gap: {max_gap_row['gap']:.3f}")
print(f"  Effective Rank: {max_gap_row['eff_rank']:.3f}")
print(f"  Penalty: {(1 - max_gap_row['correction'])*100:.1f}%")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Color scheme
C = {'orig':'#4a9eff', 'adj':'#ff6b6b', 'rank':'#a78bfa',
     'factor':'#fb923c', 'event':'#facc15', 'normal':'#34d399'}


# ==================== Figure 1: Health Score ====================
fig1, ax1 = plt.subplots(figsize=(12, 5), facecolor='#0f1117')
ax1.set_facecolor('#0f1117')

ax1.plot(results.index, results['health_orig'], color=C['orig'], lw=2, 
         label='Original Health Score')
ax1.plot(results.index, results['health_adj'], color=C['adj'], lw=2, 
         linestyle='--', label='RMT-Adjusted Health Score')

ax1.fill_between(results.index, results['health_orig'], results['health_adj'],
                 where=results['health_adj'] < results['health_orig'], 
                 color=C['adj'], alpha=0.18)

ax1.axhline(0.5, color='#888', lw=1, linestyle=':', label='Risk Threshold (0.5)')
ax1.axvline(EVENT_DATE, color=C['event'], lw=2, linestyle='--')

ax1.text(EVENT_DATE + pd.Timedelta(days=5), 0.08, 'Flash Crash\n2025-10-10',
         color=C['event'], fontsize=10, fontweight='bold', va='bottom')

ax1.set_ylabel('Health Score [0–1]', color='#cccccc')
ax1.set_title('Figure 1: LlamaRisk Market Health Score\nOriginal vs RMT-Adjusted',
              color='white', fontsize=13, pad=15)

ax1.legend(facecolor='#1a1a2e', edgecolor='#444', labelcolor='#cccccc', loc='upper left')
ax1.grid(True, alpha=0.25, color='#444')
ax1.xaxis_date()
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=30, ha='right', color='#cccccc')
ax1.tick_params(colors='#cccccc')

plt.tight_layout()
plt.savefig('fig1_health_score.png', dpi=160, facecolor='#0f1117', bbox_inches='tight')
plt.show()


# ==================== Figure 2: Effective Rank ====================
fig2, ax2 = plt.subplots(figsize=(12, 4), facecolor='#0f1117')
ax2.set_facecolor('#0f1117')

ax2.plot(results.index, results['eff_rank'], color=C['rank'], lw=2, 
         label='Effective Rank')
ax2.axhline(eff_rank_normal, color='#888', lw=1.5, linestyle='--',
            label=f'Normal Baseline ({eff_rank_normal:.2f})')
ax2.axvline(EVENT_DATE, color=C['event'], lw=2, linestyle='--')

ax2.set_ylabel('Effective Rank', color='#cccccc')
ax2.set_title('Figure 2: Effective Rank (Lower = Higher Correlation)',
              color='white', fontsize=13, pad=15)

ax2.legend(facecolor='#1a1a2e', edgecolor='#444', labelcolor='#cccccc')
ax2.grid(True, alpha=0.25, color='#444')
ax2.xaxis_date()
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right', color='#cccccc')
ax2.tick_params(colors='#cccccc')

plt.tight_layout()
plt.savefig('fig2_eff_rank.png', dpi=160, facecolor='#0f1117', bbox_inches='tight')
plt.show()


# ==================== Figure 3: Correction Factor ====================
fig3, ax3 = plt.subplots(figsize=(12, 4), facecolor='#0f1117')
ax3.set_facecolor('#0f1117')

ax3.plot(results.index, results['correction'], color=C['factor'], lw=2, 
         label='Correction Factor')
ax3.axhline(1.0, color='#888', lw=1, linestyle='--')
ax3.fill_between(results.index, results['correction'], 1.0,
                 where=results['correction'] < 0.97, color='red', alpha=0.25)
ax3.axvline(EVENT_DATE, color=C['event'], lw=2, linestyle='--')

ax3.set_ylabel('Correction Factor', color='#cccccc')
ax3.set_title('Figure 3: RMT Correction Factor (When <1 = Risk Penalty)',
              color='white', fontsize=13, pad=15)

ax3.legend(facecolor='#1a1a2e', edgecolor='#444', labelcolor='#cccccc')
ax3.grid(True, alpha=0.25, color='#444')
ax3.xaxis_date()
ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax3.xaxis.get_majorticklabels(), rotation=30, ha='right', color='#cccccc')
ax3.tick_params(colors='#cccccc')

plt.tight_layout()
plt.savefig('fig3_correction.png', dpi=160, facecolor='#0f1117', bbox_inches='tight')
plt.show()


# ==================== Figure 4: Dimension Heatmap ====================
fig4, ax4 = plt.subplots(figsize=(12, 6), facecolor='#0f1117')
ax4.set_facecolor('#0f1117')

dim_cols = [f's_{d}' for d in DIM_NAMES]
dim_matrix = results[dim_cols].T.values

im = ax4.imshow(dim_matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1,
                extent=[mdates.date2num(results.index[0]),
                        mdates.date2num(results.index[-1]), -0.5, 7.5])

ax4.set_yticks(range(8))
labels_ordered = [d.replace('_', ' ').title() for d in reversed(DIM_NAMES)]
ax4.set_yticklabels(labels_ordered, color='#cccccc')

ax4.axvline(mdates.date2num(EVENT_DATE), color=C['event'], lw=2, linestyle='--')

ax4.set_title('Figure 4: Individual Dimension Scores (Red=Risk, Green=Healthy)',
              color='white', fontsize=13, pad=15)

cb = plt.colorbar(im, ax=ax4, fraction=0.02, pad=0.01)
cb.ax.tick_params(colors='#cccccc')

ax4.xaxis_date()
ax4.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax4.xaxis.get_majorticklabels(), rotation=30, ha='right', color='#cccccc')
ax4.tick_params(colors='#cccccc')

plt.tight_layout()
plt.savefig('fig4_heatmap.png', dpi=160, facecolor='#0f1117', bbox_inches='tight')
plt.show()

print("Four dark-themed charts have been saved!")

In [ ]:
# ── Verify Event Details from July–September 2024 ─────────────────────────────

aug2024_start = pd.Timestamp('2024-07-01')
aug2024_end = pd.Timestamp('2024-10-01')
aug_window = results.loc[aug2024_start:aug2024_end]

print("Key Metrics for July–September 2024 Window:")
print(f"{'Date':12s} {'Original Health':>12s} {'Adjusted Health':>12s} "
      f"{'Eff. Rank':>10s} {'Correction':>9s}")
print("-"*55)

# Print every 3 days for readability
for date, row in aug_window.iloc[::3].iterrows():
    print(f"{str(date.date()):12s} {row['health_orig']:12.3f} "
          f"{row['health_adj']:12.3f} {row['eff_rank']:10.3f} "
          f"{row['correction']:9.3f}")

# Find the date with the largest gap between original and adjusted health
aug_window['gap'] = aug_window['health_orig'] - aug_window['health_adj']
max_gap_idx = aug_window['gap'].idxmax()
max_gap_row = aug_window.loc[max_gap_idx]

print(f"\nLargest Gap Point: {max_gap_idx.date()}")
print(f"  Original Health: {max_gap_row['health_orig']:.3f}")
print(f"  Adjusted Health: {max_gap_row['health_adj']:.3f}")
print(f"  Gap: {max_gap_row['gap']:.3f}")
print(f"  Effective Rank: {max_gap_row['eff_rank']:.3f}")
print(f"  Penalty: {(1 - max_gap_row['correction'])*100:.1f}%")